# 🚀 FastAPI Endpoints — Health, Prediction & Clinical Report

> **Module 5 of 8** — The Cervical Cancer Prediction Service

---

## 🎯 What This Module Does

This notebook demonstrates the **REST API endpoints** that expose the ML model to the world. Built with **FastAPI**, the service provides:

| Method | Endpoint | Purpose |
|--------|----------|---------|
| `GET` | `/health` | Service health & model metadata |
| `GET` | `/models` | List all registered models |
| `POST` | `/predict` | Predict cervical cancer risk |
| `POST` | `/predict/report` | Predict + generate clinical report |
| `GET` | `/docs` | Interactive Swagger UI |

## 🏗️ Request → Response Flow

```
Client ──▶ POST /predict ──▶ Pydantic Validation ──▶ Preprocess ──▶ Model ──▶ Response
   │                                                                          │
   └──────────────────────────────────────────────────────────────────────────┘
```

## 💡 Why REST API Design Matters

| Principle | Implementation |
|-----------|----------------|
| **Self-documenting** | Swagger UI at `/docs` |
| **Structured errors** | 422 for validation, 404 for missing model, 500 for server errors |
| **Dynamic model selection** | `?model=Random_Forest` query parameter |
| **Confidence scores** | Probability of positive class returned |
| **Threshold application** | `PREDICTION_THRESHOLD` determines final class |

---

## 🧠 Skills Demonstrated

- ✅ **FastAPI** — modern, async Python web framework
- ✅ **REST API Design** — clean, predictable endpoints
- ✅ **OpenAPI/Swagger** — auto-generated interactive docs
- ✅ **Error Handling** — structured, meaningful error responses

In [ ]:
# ── Add the api directory to sys.path so 'app' module can be found ──
import sys
from pathlib import Path

_candidates = [
    Path.cwd() / "cervical-cancer-mlops" / "api",
    Path.cwd().parent / "api",
    Path.cwd() / "api",
]
api_dir = next((p for p in _candidates if (p / "app").exists()), None)
if api_dir is None:
    raise RuntimeError(
        "Could not locate the 'api/app' package. "
        "Expected it under one of: " + ", ".join(map(str, _candidates))
    )
if str(api_dir) not in sys.path:
    sys.path.insert(0, str(api_dir))
print(f"✅ Added {api_dir} to sys.path")


In [ ]:
# ── Import the routers ──
from app.routers import health, predict, report

print("✅ Imported all routers:")
print(f"   - health.router  ({len(health.router.routes)} routes)")
print(f"   - predict.router ({len(predict.router.routes)} routes)")
print(f"   - report.router  ({len(report.router.routes)} routes)")

# Show all routes
print("\n📋 Registered routes:")
for r in health.router.routes + predict.router.routes + report.router.routes:
    methods = ",".join(r.methods) if hasattr(r, "methods") else "?"
    print(f"   {methods:<8} {r.path}")

In [ ]:
# ── Demonstrate the /health endpoint logic ──
from app.model_loader import load_default_model, get_default_model
from app.preprocessor import load_feature_columns
from app.config import settings

# Load the default model (as the lifespan would at startup)
load_default_model()
_, model_version, model_stage, _ = get_default_model()
features = load_feature_columns()

print("🏥 HEALTH CHECK RESPONSE")
print("=" * 50)
print(f"status:         ok")
print(f"model_name:     {settings.MODEL_NAME}")
print(f"model_version:  {model_version}")
print(f"model_stage:    {model_stage}")
print(f"features_count: {len(features)}")

In [ ]:
# ── Demonstrate the /predict endpoint logic ──
from app.model_loader import get_model, get_prediction_confidence
from app.preprocessor import preprocess

# Sample patient (low-risk)
sample_patient = {
    "Age": 25, "Number of sexual partners": 1, "First sexual intercourse": 22,
    "Num of pregnancies": 0, "Smokes": 0, "Smokes (years)": 0,
    "Smokes (packs/year)": 0.0, "Hormonal Contraceptives": 0,
    "Hormonal Contraceptives (years)": 0.0, "IUD": 0, "IUD (years)": 0.0,
    "STDs": 0, "STDs (number)": 0, "STDs:condylomatosis": 0,
    "STDs:cervical condylomatosis": 0, "STDs:vaginal condylomatosis": 0,
    "STDs:vulvo-perineal condylomatosis": 0, "STDs:syphilis": 0,
    "STDs:pelvic inflammatory disease": 0, "STDs:genital herpes": 0,
    "STDs:molluscum contagiosum": 0, "STDs:AIDS": 0, "STDs:HIV": 0,
    "STDs:Hepatitis B": 0, "STDs:HPV": 0, "STDs: Number of diagnosis": 0,
    "STDs: Time since first diagnosis": 0.0, "STDs: Time since last diagnosis": 0.0,
    "Dx:Cancer": 0, "Dx:CIN": 0, "Dx:HPV": 0, "Dx": 0,
    "Hinselmann": 0, "Schiller": 0, "Citology": 0
}

# Load default model and predict
pred_model, model_version, model_stage, model_name = get_default_model()
features_array = preprocess(sample_patient)
prediction = pred_model.predict(features_array)
pred_class = int(prediction[0])
confidence = get_prediction_confidence(pred_model, features_array)

# Apply threshold
final_class = 1 if confidence >= settings.PREDICTION_THRESHOLD else 0 if confidence is not None else pred_class

print("🔮 PREDICTION RESPONSE")
print("=" * 50)
print(f"model_name:       {model_name}")
print(f"model_version:    {model_version}")
print(f"prediction:       {final_class}")
print(f"prediction_label: {'Positive' if final_class == 1 else 'Negative'}")
print(f"confidence:       {confidence:.4f}")
print(f"threshold:        {settings.PREDICTION_THRESHOLD}")

In [ ]:
# ── Demonstrate dynamic model selection ──
# Predict with a different model (Random_Forest)
rf_model, rf_version, rf_stage, rf_name = get_model("Random_Forest")
rf_pred = rf_model.predict(features_array)
rf_conf = get_prediction_confidence(rf_model, features_array)

print("🔄 DYNAMIC MODEL SWITCHING")
print("=" * 50)
print(f"Model:       {rf_name} v{rf_version} ({rf_stage})")
print(f"Prediction:  {int(rf_pred[0])}")
print(f"Confidence:  {rf_conf:.4f}")
print(f"\nCompare with default model (Random Forest):")
print(f"  Random Forest: {final_class} (conf: {confidence:.4f})")
print(f"  Random_Forest: {int(rf_pred[0])} (conf: {rf_conf:.4f})")

---

## ✅ Summary

The FastAPI endpoints provide a **clean, self-documenting REST interface** to the ML model. With Swagger UI, structured errors, dynamic model selection, and confidence scores, this API is straightforward to integrate with.

**Next:** [Module 6 — Clinical Report Generation](api_06_report_generator.ipynb)